# Retrieval and prompt refinement

The Part 6 evaluation found that only 8 of 25 passages were directly useful across the five answerable questions. Common issues were topic-only matches, bibliography fragments and repeated sources. This notebook tests a revised ranking heuristic on the saved passages and provides an extension to the existing hybrid retriever. The relevance labels are preliminary and AI-assisted.


## 1. Setup

Import the ranking and prompt functions. The live sections require the existing Part 5 retriever and, for generation, its LLM in the same kernel.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))
import retrieval_refinement as refinement

OUT = HERE / "refinement_outputs"
OUT.mkdir(exist_ok=True)
print("Refinement functions loaded; output folder:", OUT)

## 2. Offline ranking comparison

Reorder the saved top-five passages and compare nDCG@5 using the preliminary relevance labels. The candidate set stays fixed, so this measures ordering rather than retrieval coverage.


In [ ]:
# Locate the saved Part 6 evaluation files.
BASE = Path("../data_processed/rag_final_evaluation")
if not BASE.exists():
    BASE = Path("data_processed/rag_final_evaluation")

passages_path, questions_path = BASE / "retrieved_passages.csv", BASE / "questions.csv"
if passages_path.exists() and questions_path.exists():
    passages = pd.read_csv(passages_path).fillna("")
    questions = pd.read_csv(questions_path).fillna("")
    merged = passages.merge(questions[["id", "question", "type"]], on="id", validate="many_to_one")
    # Preserve the original rank for comparison.
    reranked_rows = []
    for qid, group in merged.groupby("id", sort=False):
        question = str(group.iloc[0]["question"])
        ordered = refinement.reorder_saved_top5(question, group.to_dict("records"))
        for position, record in enumerate(ordered, 1):
            row = dict(record)
            row["new_rank"] = position
            row["relevance_cue_score"] = refinement.lexical_evidence_score(question, row["passage"], row["title"])
            row["low_value_flag"] = refinement.is_low_value_passage(row["passage"])
            reranked_rows.append(row)
    reranked = pd.DataFrame(reranked_rows)
    reranked.to_csv(OUT / "offline_reranked_passages.csv", index=False)
    display(reranked[["id", "rank", "new_rank", "title", "relevance_cue_score", "low_value_flag"]])
else:
    print("October CSV files were not found at", BASE)
    print("Clone/download the existing part-6-results files, then run this cell.")

In [ ]:
# Compare ranks using the preliminary relevance labels.
labels_path = HERE / "preliminary_relevance_labels.csv"
if "reranked" in globals() and labels_path.exists():
    labels = pd.read_csv(labels_path)
    scores = []
    for qid, group in reranked.groupby("id", sort=False):
        if group.iloc[0]["type"] != "answerable":
            continue
        review = labels[labels["id"] == qid]
        original = group.merge(review, on=["id", "rank"], validate="one_to_one").sort_values("rank")
        changed = original.sort_values("new_rank")
        before = refinement.ndcg(original["relevance"].tolist())
        after = refinement.ndcg(changed["relevance"].tolist())
        scores.append({"id": qid, "baseline_ndcg_at_5": before,
                       "offline_reranked_ndcg_at_5": after,
                       "directly_useful_in_candidate_set": int((original["relevance"] == 2).sum())})
    offline_comparison = pd.DataFrame(scores)
    offline_comparison.to_csv(OUT / "offline_ranking_diagnostic.csv", index=False)
    display(offline_comparison)
    print("Ranking comparison complete (fixed candidate set).")
else:
    print("No compatible saved passages or labels available; offline diagnostic skipped.")

## 3. Hybrid retrieval and reranking

Reuse `fusion_retriever` from Part 5. Rerank up to 30 candidates using a cross-encoder and lexical evidence cues, filter likely bibliography or checklist fragments, and retain at most two passages per paper. Model weights may be downloaded on the first run. Live retrieval has not yet been evaluated.


In [ ]:
RUN_LIVE_RETRIEVAL = False  # Change to True only after Part 5 has been run.

if RUN_LIVE_RETRIEVAL:
    if "fusion_retriever" not in globals():
        raise RuntimeError("Run Part 5 first so fusion_retriever is available")
    questions = pd.read_csv(questions_path)
    live_rows = []
    live_hits = {}
    for q in questions.itertuples(index=False):
        selected, audit = refinement.refine_existing_pipeline(q.question, fusion_retriever, top_k=5)
        live_hits[q.id] = selected
        for item in audit:
            live_rows.append({"id": q.id, "question": q.question, **item})
    live = pd.DataFrame(live_rows)
    live.to_csv(OUT / "hybrid_crossencoder_ranked_sources.csv", index=False)
    display(live)
    print("Saved live rankings for review.")
else:
    print("Live retrieval disabled.")

## 4. Evidence-grounded prompt

Use numbered source references, separate research findings from possible business applications, and acknowledge missing evidence. The KPMG savings question checks a targeted safeguard for claims that public research cannot establish.


In [ ]:
if "live_hits" in globals() and live_hits.get("Q3"):
    example_question = questions.loc[questions.id == "Q3", "question"].iloc[0]
    example_prompt = refinement.make_prompt(example_question, live_hits["Q3"])
    print(example_prompt[:4000])
else:
    print("Enable live retrieval to preview the prompt with sources.")

q6 = "What exact financial savings has KPMG achieved from this student prototype?"
print("Missing-evidence safeguard:", refinement.should_abstain_missing_evidence(q6))
print("Grounded response:", refinement.answer_with_existing_llm(q6, [], llm=None))

## 5. Answer generation

Generation is disabled by default. Enable it after initialising the retriever and LLM to export answers for review. This may use API quota. Check factual support and citations against the retrieved passages.


In [ ]:
RUN_LIVE_GENERATION = False
if RUN_LIVE_GENERATION:
    if "live_hits" not in globals() or "llm" not in globals():
        raise RuntimeError("Run the retrieval cell and load Part 5's llm first")
    generated = []
    for q in questions.itertuples(index=False):
        answer = refinement.answer_with_existing_llm(q.question, live_hits.get(q.id, []), llm)
        generated.append({"id":q.id, "question":q.question, "answer":answer})
    pd.DataFrame(generated).to_csv(OUT / "refined_answers_for_review.csv", index=False)
    print("Saved generated answers; factual review is still required.")
else:
    print("No LLM calls made.")

## Findings and follow-up

The saved-results comparison increased mean nDCG@5 from 0.669 to 0.907 across five questions. The number of directly useful passages remained 8/25. The first question still lacks directly useful evidence despite its improved ordering. Live retrieval, independent relevance review and generated-answer evaluation remain outstanding. See `REFINEMENT_REPORT.md` for the results.
